# Notebook 05 — Bias & Fairness Analysis
**Project**: IndicSenti — Multilingual Indian Sentiment Analysis  
Runs BiasChecker, calibration analysis, and demographic parity evaluation.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.calibration import calibration_curve

PALETTE = {
    'mint': '#F1F6F4', 'gold': '#FFC801', 'teal': '#114C5A',
    'sage': '#D9E8E2', 'orange': '#FF9932', 'dark': '#172B36',
}
plt.rcParams.update({
    'figure.facecolor': PALETTE['mint'], 'axes.facecolor': PALETTE['mint'],
    'axes.titlecolor': PALETTE['teal'], 'axes.labelcolor': PALETTE['dark'],
    'text.color': PALETTE['dark'], 'grid.color': PALETTE['sage'],
    'figure.dpi': 120,
})

DATA_DIR   = Path('../data')
MODEL_DIR  = Path('../models/indicsenti-lora')
REPORT_DIR = Path('../reports')
REPORT_DIR.mkdir(exist_ok=True)
print('Setup complete.')

In [ ]:
# ── Load model and test data ──────────────────────────────────
import torch
from transformers import AutoTokenizer
from peft import PeftModel
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
base_model = AutoModelForSequenceClassification.from_pretrained(
    'ai4bharat/indic-bert', num_labels=3
)
model = PeftModel.from_pretrained(base_model, str(MODEL_DIR)).to(device)
model.eval()
print('Model loaded on:', device)

test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(f'Test samples: {len(test_df):,}')

In [ ]:
# ── Batch predictions with probabilities ─────────────────────
import torch.nn.functional as F

def predict_batch(texts, batch_size=64):
    all_probs, all_preds = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=128, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        all_preds.extend(probs.argmax(axis=1).tolist())
    return np.vstack(all_probs), all_preds

texts  = test_df['text'].astype(str).tolist()
labels = test_df['label'].tolist()

print('Running inference on test set...')
probs_all, preds = predict_batch(texts)
print(f'Done. Overall Macro-F1: {__import__("sklearn.metrics",fromlist=["f1_score"]).f1_score(labels, preds, average="macro"):.4f}')

In [ ]:
# ── Fig 12: Per-language F1 ───────────────────────────────────
from sklearn.metrics import f1_score

LANGS = ['Hindi', 'Tamil', 'Bengali', 'Marathi', 'Telugu', 'English']
LANG_COLORS = [PALETTE['teal'], PALETTE['gold'], PALETTE['orange'],
               '#7CB9C4', '#F9A825', PALETTE['sage']]

lang_f1s = {}
for lang in LANGS:
    mask = test_df['language'] == lang
    if mask.sum() == 0: continue
    lang_f1s[lang] = f1_score(
        [labels[i] for i, m in enumerate(mask) if m],
        [preds[i]  for i, m in enumerate(mask) if m],
        average='macro'
    )

fig, ax = plt.subplots(figsize=(9, 4))
sorted_langs = sorted(lang_f1s, key=lang_f1s.get, reverse=True)
f1_vals = [lang_f1s[l] for l in sorted_langs]
colors  = [LANG_COLORS[LANGS.index(l)] for l in sorted_langs]

bars = ax.barh(sorted_langs, f1_vals, color=colors, edgecolor='white')
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontweight='bold', color=PALETTE['dark'])
ax.set_xlim(0.7, 0.95)
ax.set_xlabel('Macro-F1')
ax.set_title('Figure 12 — Per-Language Macro-F1')
ax.grid(axis='x', alpha=0.4)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../reports/fig12_per_language_f1.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 13: Calibration curves ────────────────────────────────
label_names = ['Positive', 'Neutral', 'Negative']
class_colors = [PALETTE['teal'], PALETTE['gold'], PALETTE['orange']]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Figure 13 — Calibration Curves (Reliability Diagrams)',
             color=PALETTE['teal'], fontweight='bold')

ece_scores = []
for cls_idx, (ax, lname, color) in enumerate(zip(axes, label_names, class_colors)):
    binary_labels = (np.array(labels) == cls_idx).astype(int)
    cls_probs     = probs_all[:, cls_idx]

    frac_pos, mean_pred = calibration_curve(binary_labels, cls_probs, n_bins=10)

    # ECE
    bins = np.linspace(0, 1, 11)
    ece = 0.0
    for i in range(len(bins)-1):
        mask = (cls_probs >= bins[i]) & (cls_probs < bins[i+1])
        if mask.sum() > 0:
            ece += np.abs(cls_probs[mask].mean() - binary_labels[mask].mean()) * mask.mean()
    ece_scores.append(ece)

    ax.plot([0,1], [0,1], 'k--', linewidth=1.5, label='Perfect calibration')
    ax.plot(mean_pred, frac_pos, color=color, linewidth=2.5, marker='o', label=f'ECE={ece:.3f}')
    ax.fill_between(mean_pred, frac_pos, mean_pred,
                    alpha=0.15, color=color, label='Calibration gap')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(lname)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

print(f'ECE scores — Positive: {ece_scores[0]:.4f}, Neutral: {ece_scores[1]:.4f}, Negative: {ece_scores[2]:.4f}')
print(f'Mean ECE: {np.mean(ece_scores):.4f}')

plt.tight_layout()
plt.savefig('../reports/fig13_calibration_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 14: Bias dimensions radar chart ─────────────────────
# Simulate bias checker output (replace with BiasChecker.run_full_audit() in real run)
from backend.bias.checker import BiasChecker

def infer_fn(text_list):
    _, preds_list = predict_batch(text_list)
    return preds_list

checker = BiasChecker(model=None, tokenizer=None, inference_fn=infer_fn)
# Run with test data
report = checker.run_full_audit(texts[:500], labels[:500])
print('Bias Report:')
print(f'  Overall bias score: {report.overall_bias_score:.4f}')
print(f'  Flags: {report.bias_flags}')

In [ ]:
# ── Fig 15: Bias gap bar chart ───────────────────────────────
# Mock values for offline execution — replace with report.* in real run
bias_dims = ['Gender', 'Regional', 'Script', 'Brand', 'Class Imb.']
bias_gaps  = [0.031, 0.048, 0.062, 0.044, 0.044]  # from paper
threshold  = 0.05

fig, ax = plt.subplots(figsize=(8, 4))
colors = [PALETTE['orange'] if g > threshold else PALETTE['teal'] for g in bias_gaps]
bars = ax.bar(bias_dims, bias_gaps, color=colors, edgecolor='white', width=0.5)

ax.axhline(threshold, color=PALETTE['gold'], linestyle='--', linewidth=2,
           label=f'Threshold ({threshold})')

for bar, val, color in zip(bars, bias_gaps, colors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.3f}', ha='center', fontweight='bold', color=PALETTE['dark'])

ax.set_ylabel('Bias Gap')
ax.set_title('Figure 14 — Bias Gaps by Dimension (lower is better)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

# Annotations
script_idx = bias_dims.index('Script')
ax.annotate('⚠️ Flagged\n(Telugu gap)', xy=(script_idx, bias_gaps[script_idx]),
            xytext=(script_idx+0.4, bias_gaps[script_idx]+0.006),
            arrowprops=dict(arrowstyle='->', color=PALETTE['orange']),
            color=PALETTE['orange'], fontsize=9)

plt.tight_layout()
plt.savefig('../reports/fig14_bias_gaps.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Full classification report ───────────────────────────────
print('='*60)
print('Full Classification Report (Test Set)')
print('='*60)
print(classification_report(labels, preds, target_names=['Positive', 'Neutral', 'Negative']))